# v42 — 18 экспериментов с OSNet R1

Основной Mac, существующее окружение **Car-classification-MSK/.venv**. Restart Kernel → Run All.
v41 продолжает работать отдельно на Mac mini; его папку не обновлять во время запуска.

Продолжаем один R1 step800: три P/K × два loss × три LR. Все варианты получают одинаковое количество изображений. Сохраняем все четыре оценённых checkpoint, выбираем лучшие конкретные веса, а не среднее по seed.
Без NiVe-обучения, новых загрузок, изменения bbox/validation, порогов или MVP. Нет ограничения времени и автоматического переключения продукта. Это primary-development поиск; его mAP не равен метрике релиза v25.

In [4]:
import os, sys, subprocess
from pathlib import Path
env = {'ORT_DISABLE_TELEMETRY': '1', 'OPENBLAS_NUM_THREADS': '1', 'OMP_NUM_THREADS': '4',
       'PYTORCH_ENABLE_MPS_FALLBACK': '0', 'PYTORCH_MPS_FAST_MATH': '0'}
if 'torch' in sys.modules and any(os.environ.get(k) != v for k, v in env.items()):
    raise RuntimeError('Restart Kernel: параметры окружения задаются до импорта torch')
os.environ.update(env)
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'training/osnet_r1_search.py').is_file()), None)
if ROOT is None:
    raise RuntimeError('Открой notebook внутри исследовательского Car-classification-MSK')
os.chdir(ROOT)
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
if sys.prefix == sys.base_prefix:
    raise RuntimeError('Выбери .venv проекта, не системный Python')
RUN_NAME = 'r1_search_v1'
DEVICE = 'mps'
INPUTS = ROOT / 'research_transfer/v41_inputs'
if not (INPUTS / 'inputs.json').is_file():
    raise FileNotFoundError('Нужна уже подготовленная research_transfer/v41_inputs; ничего автоматически не скачивается')
print('Python:', sys.executable, '\nПроект:', ROOT, '\nВходы:', INPUTS, '\nRun:', RUN_NAME, flush=True)

Python: /Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/.venv/bin/python 
Проект: /Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK 
Входы: /Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/research_transfer/v41_inputs 
Run: r1_search_v1


In [5]:
print('ЭТАП 1/3 — компонентные проверки (без полного обучения)', flush=True)
subprocess.run([sys.executable, '-m', 'pytest', '-q', 'tests/test_osnet_r1_search.py'], check=True)

ЭТАП 1/3 — компонентные проверки (без полного обучения)
.................ssssss                                                  [100%]
17 passed, 6 skipped in 3.46s


CompletedProcess(args=['/Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/.venv/bin/python', '-m', 'pytest', '-q', 'tests/test_osnet_r1_search.py'], returncode=0)

In [6]:
print('ЭТАП 2/3 — preflight, контроль и 18 экспериментов. Общего лимита времени нет.', flush=True)
from training.osnet_r1_search import run, VARIANT
results = run(INPUTS, RUN_NAME, DEVICE)

ЭТАП 2/3 — preflight, контроль и 18 экспериментов. Общего лимита времени нет.
MPS FP32 | allocator limit 13.6 GiB | no fallback | disk minimum 3 GiB, recommended 5–6 GiB
PREFLIGHT: 740 train-ID, 185 holdout-ID | mps  | frozen primary only
CONTROL: B0
FEATURES 16/1036
FEATURES 272/1036
FEATURES 528/1036
FEATURES 784/1036
FEATURES 1036/1036
CONTROL: R1_20260915
FEATURES 16/1036
FEATURES 272/1036
FEATURES 528/1036
FEATURES 784/1036
FEATURES 1036/1036
CONTROL: R1_20260916
FEATURES 16/1036
FEATURES 272/1036
FEATURES 528/1036
FEATURES 784/1036
FEATURES 1036/1036
CONTROL: R1_20260917
FEATURES 16/1036
FEATURES 272/1036
FEATURES 528/1036
FEATURES 784/1036
FEATURES 1036/1036

TRIAL 1/18: R01_p16k2_supcon_lr0.1 | mps | lr=1.1940564e-05
R01_p16k2_supcon_lr0.1 | step 25/200 | exposure 800/6400 | passes 0.17 | loss 1.2549 | time 0.4 min | stage ETA 2.8 min
R01_p16k2_supcon_lr0.1 | step 50/200 | exposure 1600/6400 | passes 0.35 | loss 1.3446 | time 0.8 min | stage ETA 2.4 min
R01_p16k2_supcon_lr0.1 |

In [7]:
print('ЭТАП 3/3 — итоговый отчёт', flush=True)
from IPython.display import Markdown, display
display(Markdown((VARIANT / 'runs' / RUN_NAME / 'REPORT.md').read_text(encoding='utf-8')))
print('Архив для анализа:', VARIANT / f'{RUN_NAME}_analysis.zip')
print('Веса сохраняются в runs; компактный архив их не содержит. v25 не изменён.')

ЭТАП 3/3 — итоговый отчёт


# v42 — целевой поиск OSNet R1

Статус: complete.

Завершено 18/18 конфигураций.
C_primary = B0_208 step800 + equal3 R1_256 step800, cosine 50/50; NOT release v25

Продолжение одного R1 step800 с новым AdamW. Не обучение с нуля и не новый независимый тест.
Best concrete checkpoint: без усреднения качества по seed. v25 и v41 не изменены.
Порог/F1/TNR не подбираются; это исследование ranking, не готовая замена MVP.

| Trial | Доп. предъявления | Single raw | Single graph | Add G2 | Replace R1 | Δ system |
|---|---:|---:|---:|---:|---:|---:|
| R03_p16k2_supcon_lr1 | 12800 | 0.816520 | 0.829962 | 0.849818 | 0.852150 | +0.002696 |
| R09_p16k4_supcon_lr1 | 6400 | 0.821343 | 0.834297 | 0.850328 | 0.845714 | +0.000874 |
| R01_p16k2_supcon_lr0.1 | 12800 | 0.830361 | 0.839265 | 0.849868 | 0.847135 | +0.000414 |
| R12_p16k4_soft_triplet_lr1 | 6400 | 0.804171 | 0.815365 | 0.849811 | 0.842762 | +0.000356 |
| R02_p16k2_supcon_lr0.3 | 12800 | 0.829200 | 0.839065 | 0.849791 | 0.844347 | +0.000337 |
| R01_p16k2_supcon_lr0.1 | 6400 | 0.830703 | 0.844160 | 0.849781 | 0.845581 | +0.000326 |
| R03_p16k2_supcon_lr1 | 25600 | 0.825506 | 0.838382 | 0.849646 | 0.844426 | +0.000191 |
| R01_p16k2_supcon_lr0.1 | 25600 | 0.832768 | 0.842958 | 0.849643 | 0.846372 | +0.000189 |
| unchanged_R1 | 0 | 0.832842 | 0.844099 | 0.849593 | 0.849454 | +0.000138 |
| R07_p16k4_supcon_lr0.1 | 6400 | 0.833094 | 0.842183 | 0.849584 | 0.846210 | +0.000130 |
| R11_p16k4_soft_triplet_lr0.3 | 6400 | 0.825267 | 0.838870 | 0.849573 | 0.846416 | +0.000119 |
| R09_p16k4_supcon_lr1 | 12800 | 0.818756 | 0.827250 | 0.849498 | 0.841433 | +0.000043 |
| R08_p16k4_supcon_lr0.3 | 6400 | 0.831537 | 0.843203 | 0.849438 | 0.844760 | -0.000016 |
| R13_p32k2_supcon_lr0.1 | 6400 | 0.831211 | 0.841282 | 0.849405 | 0.845528 | -0.000049 |
| R08_p16k4_supcon_lr0.3 | 25600 | 0.827696 | 0.832199 | 0.849395 | 0.846817 | -0.000060 |
| R14_p32k2_supcon_lr0.3 | 6400 | 0.834937 | 0.844443 | 0.849361 | 0.847801 | -0.000094 |
| R01_p16k2_supcon_lr0.1 | 51200 | 0.830740 | 0.844397 | 0.849323 | 0.846035 | -0.000131 |
| R15_p32k2_supcon_lr1 | 6400 | 0.825828 | 0.834642 | 0.849322 | 0.844750 | -0.000132 |
| R02_p16k2_supcon_lr0.3 | 6400 | 0.831052 | 0.842969 | 0.849313 | 0.841249 | -0.000142 |
| R06_p16k2_soft_triplet_lr1 | 6400 | 0.794452 | 0.805213 | 0.849260 | 0.843668 | -0.000194 |
| R02_p16k2_supcon_lr0.3 | 25600 | 0.830155 | 0.841106 | 0.849260 | 0.845336 | -0.000194 |
| R06_p16k2_soft_triplet_lr1 | 51200 | 0.814127 | 0.819893 | 0.849225 | 0.841718 | -0.000229 |
| R15_p32k2_supcon_lr1 | 12800 | 0.827888 | 0.842488 | 0.849198 | 0.843789 | -0.000257 |
| R07_p16k4_supcon_lr0.1 | 12800 | 0.829599 | 0.840253 | 0.849172 | 0.846939 | -0.000283 |
| R07_p16k4_supcon_lr0.1 | 25600 | 0.832525 | 0.842298 | 0.849142 | 0.847805 | -0.000313 |
| R13_p32k2_supcon_lr0.1 | 25600 | 0.830974 | 0.839678 | 0.849135 | 0.843915 | -0.000319 |
| R03_p16k2_supcon_lr1 | 6400 | 0.811096 | 0.825137 | 0.849122 | 0.842593 | -0.000332 |
| R06_p16k2_soft_triplet_lr1 | 12800 | 0.801073 | 0.809104 | 0.849118 | 0.844891 | -0.000336 |
| R13_p32k2_supcon_lr0.1 | 12800 | 0.831287 | 0.838040 | 0.849110 | 0.844058 | -0.000344 |
| R08_p16k4_supcon_lr0.3 | 51200 | 0.825334 | 0.834047 | 0.849110 | 0.846417 | -0.000345 |
| R07_p16k4_supcon_lr0.1 | 51200 | 0.830300 | 0.837761 | 0.849097 | 0.846653 | -0.000358 |
| R14_p32k2_supcon_lr0.3 | 12800 | 0.834188 | 0.842018 | 0.849094 | 0.845918 | -0.000361 |
| R08_p16k4_supcon_lr0.3 | 12800 | 0.828402 | 0.837567 | 0.849089 | 0.844191 | -0.000365 |
| R09_p16k4_supcon_lr1 | 51200 | 0.828029 | 0.833233 | 0.849089 | 0.844384 | -0.000365 |
| R03_p16k2_supcon_lr1 | 51200 | 0.830214 | 0.839052 | 0.849061 | 0.846902 | -0.000393 |
| R05_p16k2_soft_triplet_lr0.3 | 6400 | 0.811863 | 0.825279 | 0.849053 | 0.839549 | -0.000402 |
| R06_p16k2_soft_triplet_lr1 | 25600 | 0.813550 | 0.827832 | 0.849035 | 0.843172 | -0.000420 |
| R10_p16k4_soft_triplet_lr0.1 | 6400 | 0.828877 | 0.835338 | 0.849028 | 0.845486 | -0.000427 |
| R16_p32k2_soft_triplet_lr0.1 | 6400 | 0.823181 | 0.830430 | 0.849028 | 0.845992 | -0.000427 |
| R12_p16k4_soft_triplet_lr1 | 12800 | 0.809549 | 0.811970 | 0.848926 | 0.843815 | -0.000529 |
| R12_p16k4_soft_triplet_lr1 | 51200 | 0.809304 | 0.810722 | 0.848922 | 0.841289 | -0.000532 |
| R15_p32k2_supcon_lr1 | 25600 | 0.826445 | 0.831716 | 0.848906 | 0.844337 | -0.000549 |
| R17_p32k2_soft_triplet_lr0.3 | 12800 | 0.806268 | 0.813517 | 0.848906 | 0.843734 | -0.000549 |
| R18_p32k2_soft_triplet_lr1 | 12800 | 0.809399 | 0.823756 | 0.848903 | 0.845030 | -0.000552 |
| R10_p16k4_soft_triplet_lr0.1 | 12800 | 0.820052 | 0.830831 | 0.848903 | 0.844067 | -0.000552 |
| R13_p32k2_supcon_lr0.1 | 51200 | 0.832518 | 0.841736 | 0.848898 | 0.843108 | -0.000557 |
| R11_p16k4_soft_triplet_lr0.3 | 51200 | 0.806328 | 0.813176 | 0.848879 | 0.842790 | -0.000575 |
| R15_p32k2_supcon_lr1 | 51200 | 0.825667 | 0.831256 | 0.848871 | 0.840673 | -0.000583 |
| R14_p32k2_supcon_lr0.3 | 25600 | 0.830556 | 0.840212 | 0.848831 | 0.844541 | -0.000624 |
| R05_p16k2_soft_triplet_lr0.3 | 51200 | 0.797994 | 0.807494 | 0.848824 | 0.840321 | -0.000631 |
| R17_p32k2_soft_triplet_lr0.3 | 6400 | 0.819304 | 0.828272 | 0.848823 | 0.846067 | -0.000632 |
| R09_p16k4_supcon_lr1 | 25600 | 0.825389 | 0.830171 | 0.848820 | 0.843967 | -0.000634 |
| R12_p16k4_soft_triplet_lr1 | 25600 | 0.805096 | 0.813483 | 0.848782 | 0.841260 | -0.000672 |
| R05_p16k2_soft_triplet_lr0.3 | 25600 | 0.800152 | 0.811397 | 0.848779 | 0.840593 | -0.000676 |
| R18_p32k2_soft_triplet_lr1 | 51200 | 0.812410 | 0.823389 | 0.848769 | 0.842774 | -0.000685 |
| R04_p16k2_soft_triplet_lr0.1 | 12800 | 0.806589 | 0.816934 | 0.848736 | 0.842230 | -0.000719 |
| R04_p16k2_soft_triplet_lr0.1 | 6400 | 0.820968 | 0.829764 | 0.848715 | 0.845548 | -0.000740 |
| R14_p32k2_supcon_lr0.3 | 51200 | 0.833642 | 0.841164 | 0.848706 | 0.841508 | -0.000749 |
| R11_p16k4_soft_triplet_lr0.3 | 12800 | 0.811759 | 0.818995 | 0.848697 | 0.844316 | -0.000757 |
| R18_p32k2_soft_triplet_lr1 | 25600 | 0.805436 | 0.812026 | 0.848672 | 0.842449 | -0.000782 |
| R02_p16k2_supcon_lr0.3 | 51200 | 0.833891 | 0.843436 | 0.848652 | 0.839973 | -0.000802 |
| R05_p16k2_soft_triplet_lr0.3 | 12800 | 0.801928 | 0.810840 | 0.848649 | 0.839135 | -0.000805 |
| R16_p32k2_soft_triplet_lr0.1 | 12800 | 0.814479 | 0.825286 | 0.848647 | 0.845758 | -0.000807 |
| R17_p32k2_soft_triplet_lr0.3 | 25600 | 0.796580 | 0.806426 | 0.848646 | 0.840944 | -0.000809 |
| R17_p32k2_soft_triplet_lr0.3 | 51200 | 0.798292 | 0.804804 | 0.848646 | 0.840290 | -0.000809 |
| R11_p16k4_soft_triplet_lr0.3 | 25600 | 0.806335 | 0.815750 | 0.848616 | 0.843614 | -0.000839 |
| R10_p16k4_soft_triplet_lr0.1 | 25600 | 0.812920 | 0.822281 | 0.848593 | 0.843876 | -0.000861 |
| R04_p16k2_soft_triplet_lr0.1 | 51200 | 0.795831 | 0.807261 | 0.848570 | 0.840396 | -0.000885 |
| R18_p32k2_soft_triplet_lr1 | 6400 | 0.810062 | 0.820011 | 0.848552 | 0.841349 | -0.000903 |
| R16_p32k2_soft_triplet_lr0.1 | 25600 | 0.803556 | 0.810537 | 0.848538 | 0.844900 | -0.000917 |
| R10_p16k4_soft_triplet_lr0.1 | 51200 | 0.803451 | 0.810399 | 0.848468 | 0.844582 | -0.000986 |
| R04_p16k2_soft_triplet_lr0.1 | 25600 | 0.800494 | 0.808096 | 0.848460 | 0.840672 | -0.000995 |
| R16_p32k2_soft_triplet_lr0.1 | 51200 | 0.798526 | 0.808032 | 0.848443 | 0.843812 | -0.001012 |

Контроль: 0.849454. Лучший system: 0.852150 (replace_R1_20260915).

## Ошибки


В компактном ZIP нет весов. Сохраните runs целиком: checkpoints/features и per-query top10.
Три draws используют одни holdout-ID. Это не три независимых fold.
Одинаковые предъявления не означают одинаковое число optimizer updates.
Presentations считают robust-примеры; clean-consistency forward выполняется дополнительно.


Архив для анализа: /Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/OSNet-AIN-x1.0/variant_42_osnet_r1_search/r1_search_v1_analysis.zip
Веса сохраняются в runs; компактный архив их не содержит. v25 не изменён.


## Если запуск прервался
Не удаляй runs и не меняй RUN_NAME/конфигурацию/код. Restart Kernel → Run All продолжит с проверенных сохранений. Не запускай второй v42 на этой же машине одновременно. После обновления кода нужен новый RUN_NAME.

Прогресс — каждые 25 шагов; промежуточный REPORT.md обновляется после каждого оценённого checkpoint. Ошибки отдельных обучений остаются в отчёте, успешные результаты сохраняются. Если есть failed trials, выбор считается предварительным.

В терминале на время ночного запуска можно выполнить `caffeinate -i`; оставить окно открытым. Сон вручную и закрытие крышки могут остановить работу. Для остановки caffeinate — Ctrl+C.